# Chapter 5 - Classification

## Chapter summary
**Classification** predicts a categorical outcome (e.g. loan *paid off* vs *default*). Models usually output a **probability** of belonging to the class of interest, which is turned into a class using a **cutoff** (default 0.5). Topics:

1. **Naive Bayes** - Bayes' rule with a conditional-independence assumption.
2. **Discriminant analysis** (LDA) - finds the linear combination of predictors that best separates the classes.
3. **Logistic regression** - a generalised linear model: log-odds are linear in the predictors.
4. **Evaluating classification models** - confusion matrix, accuracy, precision, recall, specificity, ROC curve, AUC, lift.
5. **Strategies for imbalanced data** - undersampling, oversampling, up/down-weighting, synthetic data generation (SMOTE), cost-based classification.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, precision_recall_fscore_support, accuracy_score,
                             roc_curve, roc_auc_score)
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"

def get_data(name):
    """Download a dataset of the book (once) into ./data and return its path."""
    path = Path("data") / name
    path.parent.mkdir(exist_ok=True)
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + name, path)
    return path

sns.set_style("whitegrid")

loan_data = pd.read_csv(get_data("loan_data.csv.gz"))
loan_data = loan_data.drop(columns=[c for c in loan_data.columns if c.startswith("Unnamed")])
print(loan_data.shape)
loan_data.head()

## 1. Naive Bayes

**Theory.** The *exact* Bayes classifier would compute $P(Y=i\mid X_1,\dots,X_p)$ by finding records with identical predictor values - infeasible with many predictors. **Naive Bayes** assumes predictors are *conditionally independent given the class*, so

$$ P(Y=i\mid X_1,\dots,X_p) \;\propto\; P(Y=i)\prod_{j=1}^{p}P(X_j\mid Y=i). $$

The class probabilities are normalised to sum to 1. It works with **categorical** predictors (numeric ones must be binned or modelled with a Gaussian). Despite the unrealistic independence assumption, it often ranks classes well, but its probabilities can be poorly calibrated. A small smoothing term (`alpha`) avoids zero probabilities.

In [ ]:
predictors = ["purpose_", "home_", "emp_len_"]
outcome = "outcome"
X = pd.get_dummies(loan_data[predictors]).astype(float)       # one 0/1 column per category level
y = loan_data[outcome]

naive_model = MultinomialNB(alpha=0.01, fit_prior=True)
naive_model.fit(X, y)

new_loan = X.loc[0:0, :]
print("New loan:", loan_data.loc[0, predictors].to_dict())
print("Naive Bayes probabilities:", dict(zip(naive_model.classes_, naive_model.predict_proba(new_loan)[0].round(3))))

**Exact Bayes vs. naive Bayes.** The *exact* answer is simply the share of defaults among past loans that have **exactly the same** values of all three predictors. That works here because there are few categories, but with many predictors almost no record would match exactly (the data thin out - the "curse of dimensionality"), which is why the naive independence assumption is useful. Compare the two:

In [ ]:
row = loan_data.loc[0, predictors]
same = loan_data[(loan_data[predictors] == row).all(axis=1)]
print(f"Exact Bayes: {len(same)} past loans share these values; share that defaulted = {(same[outcome] == 'default').mean():.3f}")
print(f"Naive Bayes: P(default) = {naive_model.predict_proba(new_loan)[0][list(naive_model.classes_).index('default')]:.3f}")

## 2. Discriminant Analysis

**Theory.** **Linear discriminant analysis (LDA)** finds a linear combination (the *discriminant function*) of the predictors that maximises the **between-class** variance relative to the **within-class** variance. Equivalently, under the assumption that each class is multivariate normal with a *common covariance matrix*, it yields linear class boundaries. The resulting *discriminant scores* are converted to probabilities. LDA is related to logistic regression, PCA and regression, and is the historical forerunner of modern classifiers.

Data: a 3000-loan sample with `borrower_score` and `payment_inc_ratio`.

In [ ]:
loan3000 = pd.read_csv(get_data("loan3000.csv"))
loan3000.outcome = loan3000.outcome.astype("category")
predictors = ["borrower_score", "payment_inc_ratio"]
X, y = loan3000[predictors], loan3000["outcome"]

loan_lda = LinearDiscriminantAnalysis().fit(X, y)
print("LDA scalings:", pd.DataFrame(loan_lda.scalings_, index=X.columns).T.round(3))

pred = pd.DataFrame(loan_lda.predict_proba(X), columns=loan_lda.classes_)
print(pred.head())

In [ ]:
# decision boundary: the points where the linear discriminant score is 0, i.e. coef_[0]*x1 + coef_[1]*x2 + intercept_ = 0
b1, b2 = loan_lda.coef_[0]
xs = np.array([0.25, 0.8])
ys = -(b1 * xs + loan_lda.intercept_[0]) / b2
ax = sns.scatterplot(data=loan3000, x="borrower_score", y="payment_inc_ratio", hue="outcome",
                     style="outcome", alpha=0.5, s=20)
ax.plot(xs, ys, color="black")
ax.set_ylim(0, 20)
plt.title("LDA decision boundary"); plt.show()

## 3. Logistic Regression

**Theory.** Linear regression can output values outside [0, 1]. Logistic regression models the probability $p$ of the positive class through the **log-odds (logit)**:

$$ \text{odds}=\frac{p}{1-p},\qquad \log(\text{odds}) = b_0+b_1X_1+\dots+b_pX_p \;\Rightarrow\; p=\frac{1}{1+e^{-(b_0+b_1X_1+\dots+b_pX_p)}}. $$

* It is a **generalised linear model (GLM)** with a *binomial* family and *logit link*, fit by **maximum likelihood** (not least squares).
* A coefficient $b_j$ is the change in *log-odds* per unit of $X_j$; $e^{b_j}$ is the **odds ratio**.
* Assess with **deviance** ($-2\times$ log-likelihood), AIC, z-values and p-values. Residual diagnostics are similar to linear regression (partial residual plots help find non-linearity).

Here the positive class is `default` (1).

In [ ]:
predictors = ["payment_inc_ratio", "purpose_", "home_", "emp_len_", "borrower_score"]
y = (loan_data["outcome"] == "default").astype(int)
X = pd.get_dummies(loan_data[predictors], drop_first=True).astype(float)

logit = sm.GLM(y, sm.add_constant(X), family=sm.families.Binomial()).fit()
print(logit.summary())

In [ ]:
print("Odds ratios:")
print(np.exp(logit.params).round(3))

p = logit.predict(sm.add_constant(X))
print("Predicted P(default) summary:\n", p.describe().round(3))

**Interpreting the output.** Each coefficient is a change in **log-odds**, which is hard to feel; exponentiating gives an **odds ratio**. An odds ratio of 0.5 for a predictor means a one-unit increase *halves* the odds of default (holding the others fixed); a ratio of 2 doubles them; 1 means no effect. Note that odds are not probabilities: for rare events the two are close, but for common events they differ a lot (probability 0.5 = odds 1, probability 0.8 = odds 4). Because the link is non-linear, the same coefficient changes the *probability* by different amounts depending on where you start - at very low or very high probability, a change in log-odds barely moves the probability (the flat ends of the sigmoid below). Compare each predictor's odds ratio with its **z-value** and p-value in the summary to judge whether it matters; in the loan data a better `borrower_score` strongly *lowers* the odds of default while a higher `payment_inc_ratio` *raises* them.

In [ ]:
# log-odds -> probability
logodds = np.linspace(-6, 6, 100)
plt.figure(figsize=(5, 3.5))
plt.plot(logodds, 1 / (1 + np.exp(-logodds)))
plt.xlabel("log-odds"); plt.ylabel("probability"); plt.title("Logistic (sigmoid) function"); plt.show()

## 4. Evaluating Classification Models

**Theory.** Accuracy alone is misleading, especially for rare classes. The **confusion matrix** counts predictions vs. actuals:

| | Predicted positive | Predicted negative |
|---|---|---|
| **Actual positive** | TP | FN |
| **Actual negative** | FP | TN |

* **Accuracy** $=\frac{TP+TN}{N}$
* **Precision** $=\frac{TP}{TP+FP}$ - how many predicted positives are real.
* **Recall / sensitivity** $=\frac{TP}{TP+FN}$ - how many real positives are found.
* **Specificity** $=\frac{TN}{TN+FP}$ - how many real negatives are found.
* **F1** = harmonic mean of precision and recall.
* **ROC curve**: plots recall (true-positive rate) vs. 1 - specificity (false-positive rate) as the cutoff varies. **AUC** (area under the curve) summarises it: 0.5 = random, 1 = perfect.
* **Lift / gains charts** show how much better the model finds positives than random when ranking records by predicted probability - useful when only a fraction can be acted on.

In [ ]:
y_pred = (p > 0.5).astype(int)
cm = confusion_matrix(y, y_pred)
print(pd.DataFrame(cm, index=["Y = paid off", "Y = default"], columns=["Yhat = paid off", "Yhat = default"]))

prec, rec, f1, _ = precision_recall_fscore_support(y, y_pred, average="binary")
tn, fp, fn, tp = cm.ravel()
print(f"Accuracy={accuracy_score(y, y_pred):.3f}  Precision={prec:.3f}  Recall={rec:.3f}  "
      f"Specificity={tn / (tn + fp):.3f}  F1={f1:.3f}")

In [ ]:
fpr, tpr, thresholds = roc_curve(y, p)
auc = roc_auc_score(y, p)
plt.figure(figsize=(5, 4.5))
plt.plot(fpr, tpr, label=f"Logistic regression (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("1 - specificity (false positive rate)"); plt.ylabel("Recall (true positive rate)")
plt.legend(); plt.title("ROC curve"); plt.show()

**Reading the ROC curve and AUC.** Every point on the curve is one cutoff: a *low* cutoff labels many records positive (high recall, but many false alarms - top right), a *high* cutoff labels few (few false alarms, but misses - bottom left). A good model hugs the top-left corner; the diagonal is a coin toss. **AUC has a neat meaning**: it is the probability that a randomly chosen positive (default) receives a higher predicted probability than a randomly chosen negative, so it judges the model's *ranking* independently of any cutoff. Its limitation: it ignores the costs of the two error types and can look flattering on very imbalanced data - there, also check precision and recall.

**Accuracy and the rare-class problem.** If only 1 record in 100 is positive, predicting "negative" for everyone is 99% accurate yet finds no positives at all. This is why precision (how trustworthy are the alarms?) and recall (how many real cases did we catch?) are reported separately, and why the F1 score combines them.

In [ ]:
# Cumulative gains / lift chart
df = pd.DataFrame({"y": y.values, "p": p.values}).sort_values("p", ascending=False).reset_index(drop=True)
df["gain"] = df["y"].cumsum() / df["y"].sum()
df["pct"] = (df.index + 1) / len(df)
plt.figure(figsize=(5, 4))
plt.plot(df.pct, df.gain, label="Model")
plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("Fraction of records contacted (sorted by predicted probability)")
plt.ylabel("Fraction of all defaults found"); plt.legend(); plt.title("Cumulative gains"); plt.show()
print("Lift in top decile:", round(df.loc[: len(df) // 10, "y"].mean() / df["y"].mean(), 2))

## 5. Strategies for Imbalanced Data

**Theory.** When the positive class is rare (fraud, rare disease, purchases) a model that always predicts the majority class has high accuracy yet is useless. Remedies:

* **Undersampling** the majority class - fast, loses data.
* **Oversampling / bootstrapping** the minority class - may overfit duplicates.
* **Up-/down-weighting** records (class weights) in the loss function.
* **Data generation** - **SMOTE** creates synthetic minority records by interpolating between a minority record and its nearest neighbours.
* **Cost-based classification** - choose the cutoff by the relative cost of false positives vs. false negatives; use **precision-recall/AUC** instead of accuracy.

**Caveat - probabilities become biased.** Resampling or re-weighting changes the class proportions the model sees, so its predicted probabilities no longer match the real-world frequency of the class (a model trained on a 50/50 sample says "50%" for an average case that is really 2%). The *ranking* of records, and therefore AUC, survives, but if you need calibrated probabilities you must correct them afterwards (e.g. adjust the intercept, or calibrate on untouched data). Always **evaluate on a test set that keeps the original class balance** - never on resampled data - and resample **only the training part** (otherwise duplicated or synthetic records leak into the test set).

*Adaptation note:* the book uses a forest-cover dataset; to keep this notebook self-contained we create an imbalanced problem from the loan data by keeping only about 5% of defaults.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

defaults = loan_data[loan_data["outcome"] == "default"]
paid = loan_data[loan_data["outcome"] == "paid off"]
imb = pd.concat([paid, defaults.sample(frac=0.05, random_state=1)]).sample(frac=1, random_state=1)
print(f"Records: {len(imb)}   fraction of defaults: {(imb['outcome'] == 'default').mean():.2%}")

preds = ["payment_inc_ratio", "borrower_score", "dti", "revol_util", "open_acc", "annual_inc"]
X_imb = imb[preds].fillna(imb[preds].median())
y_imb = (imb["outcome"] == "default").astype(int)

# the test set keeps the ORIGINAL (imbalanced) class balance; only the training part is ever resampled
X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, test_size=0.3, stratify=y_imb, random_state=1)

def make_lr(**kw):
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, **kw))

def report(label, model, cutoff=0.5):
    proba = model.predict_proba(X_te)[:, 1]
    pred = proba > cutoff
    prec, rec, _, _ = precision_recall_fscore_support(y_te, pred, average="binary", zero_division=0)
    print(f"{label:36s} accuracy={accuracy_score(y_te, pred):.3f}  recall={rec:.3f}  precision={prec:.3f}  AUC={roc_auc_score(y_te, proba):.3f}")

print(f"A model that always says 'paid off' has accuracy {1 - y_te.mean():.3f} and recall 0.\n")
report("Baseline (no rebalancing)", make_lr().fit(X_tr, y_tr))
weighted = make_lr(class_weight="balanced").fit(X_tr, y_tr)
report("Class weights (up-weight defaults)", weighted)

train = X_tr.assign(y=y_tr)
minority, majority = train[train["y"] == 1], train[train["y"] == 0]

under = pd.concat([majority.sample(n=len(minority), random_state=1), minority])          # undersampling
report("Undersample the majority", make_lr().fit(under[preds], under["y"]))

over = pd.concat([majority, minority.sample(n=len(majority), replace=True, random_state=1)])   # oversampling (bootstrap)
report("Oversample the minority (bootstrap)", make_lr().fit(over[preds], over["y"]))

In [ ]:
# SMOTE (synthetic minority oversampling) - needs the imbalanced-learn package
try:
    from imblearn.over_sampling import SMOTE
    Xs, ys = SMOTE(random_state=1).fit_resample(X_tr, y_tr)
    print(f"SMOTE: training records {len(y_tr)} -> {len(ys)}")
    report("SMOTE (synthetic defaults)", make_lr().fit(Xs, ys))
except ImportError:
    print("Install imbalanced-learn to run SMOTE:  pip install imbalanced-learn")

**Choosing the cutoff by cost.** With rare classes a cutoff of 0.5 is rarely optimal. If a missed default (false negative) costs, say, 5 times more than a false alarm (false positive), pick the cutoff that minimises the **total expected cost** instead of maximising accuracy. Sweep the cutoff and watch the precision-recall trade-off:

In [ ]:
proba = weighted.predict_proba(X_te)[:, 1]
cost_fn, cost_fp = 5, 1
rows = []
for cut in np.arange(0.1, 0.95, 0.1):
    pred = proba > cut
    prec, rec, _, _ = precision_recall_fscore_support(y_te, pred, average="binary", zero_division=0)
    fn = int(((~pred) & (y_te.values == 1)).sum()); fp = int((pred & (y_te.values == 0)).sum())
    rows.append((round(cut, 1), prec, rec, fn, fp, cost_fn * fn + cost_fp * fp))
table = pd.DataFrame(rows, columns=["cutoff", "precision", "recall", "false neg", "false pos", "total cost"])
print(table.round(3).to_string(index=False))
print("Cheapest cutoff:", table.loc[table["total cost"].idxmin(), "cutoff"])

## Key takeaways
* Classifiers estimate class probabilities; a cutoff turns them into decisions.
* **Naive Bayes** (independent predictors), **LDA** (linear separation) and **logistic regression** (log-odds linear) are the classic, interpretable methods.
* Evaluate with the **confusion matrix, precision/recall/specificity, ROC/AUC and lift** - not accuracy alone.
* For rare classes, rebalance (under/oversample, SMOTE, weights) and tune the **cutoff** to the business costs.